### Imports

In [5]:
import os
import cv2
import torch
import numpy as np
from utils import *
import pandas as pd
import seaborn as sns
from modules import *
from PIL import Image
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import utils.for_eye_tracker as et
from scipy.signal import savgol_filter
from torchvision import models, transforms
from transformers import CLIPProcessor, CLIPModel
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights

_________

## Gaze Preprocessing and Fixation Extraction (IVT)

In [2]:
def process_gaze_to_fixations(csv_path):

    # --- LOAD ---
    df = pd.read_csv(csv_path)

    df = df.rename(columns={
        'Seconds': 'time',
        'GazeX': 'x',
        'GazeY': 'y'
    })

    df['time'] = pd.to_datetime(df['time'])

    df = df[['time', 'x', 'y']]

    # --- CLEAN ---
    df = df.drop_duplicates(subset='time')
    df = df[(df['x'] >= 0) & (df['y'] >= 0)]

    # --- PREPROCESS ---
    df = df.sort_values('time')

    df['x'] = df['x'].interpolate(limit=5)
    df['y'] = df['y'].interpolate(limit=5)

    df = df.dropna(subset=['x', 'y'])

    if len(df) > 5:
        df['x_smooth'] = savgol_filter(df['x'], 5, 2)
        df['y_smooth'] = savgol_filter(df['y'], 5, 2)
    else:
        df['x_smooth'] = df['x']
        df['y_smooth'] = df['y']

    # --- VELOCITY ---
    t_sec = pd.to_datetime(df['time']).astype('int64') / 1e9

    dt = np.diff(t_sec)
    dx = np.diff(df['x_smooth'])
    dy = np.diff(df['y_smooth'])

    dt[dt == 0] = np.nan

    velocity = np.sqrt(dx**2 + dy**2) / dt

    df['velocity'] = np.nan
    df.iloc[1:, df.columns.get_loc('velocity')] = velocity

    # --- IVT CLASSIFICATION ---
    df['event'] = 'fixation'
    df.loc[df['velocity'] > 1000, 'event'] = 'saccade'

    # --- EXTRACT FIXATIONS ---
    fixations = []
    current_fix = None

    for idx in range(len(df)):
        row = df.iloc[idx]
        if row['event'] == 'fixation':
            if current_fix is None:
                current_fix = {
                    'start_time': row['time'],
                    'x': [],
                    'y': []
                }

            current_fix['x'].append(row['x'])
            current_fix['y'].append(row['y'])

        else:
            if current_fix is not None:
                end_time = df.iloc[idx-1]['time']

                fixations.append({
                    'start_time': current_fix['start_time'],
                    'end_time': end_time,
                    'duration': (end_time - current_fix['start_time']).total_seconds(),
                    'x_mean': np.mean(current_fix['x']),
                    'y_mean': np.mean(current_fix['y'])
                })

                current_fix = None

    fixations = pd.DataFrame(fixations)

    if len(fixations) == 0:
        return None

    # --- POST FILTER ---
    fixations['duration_ms'] = fixations['duration'] * 1000
    fixations = fixations[fixations['duration_ms'] >= 100]
    fixations = fixations[fixations['duration_ms'] < 4000]

    fixations["mid_time"] = fixations["start_time"] + (
        fixations["end_time"] - fixations["start_time"]
    ) / 2

    return fixations

## Processing All Participants: Gaze → Fixations

In [3]:
data_root = "/mnt/raid/emotional_data_raquel/supp_mine/gaze"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)
        gaze_csv = os.path.join(session_path, "gaze.csv")

        if not os.path.exists(gaze_csv):
            continue

        print(f"Processing {participant} / {session}")

        try:
            fixations = process_gaze_to_fixations(gaze_csv)

            if fixations is None:
                print("No fixations found")
                continue

            # --- SAVE PATH ---
            save_dir = os.path.join(output_root, participant)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(save_dir, f"{session}_fixations.csv")

            fixations.to_csv(save_path, index=False)

            print(f"Saved → {save_path}")

        except Exception as e:
            print(f"Error in {participant}/{session}: {e}")

Processing sub-OE020 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Norrebro_fixations.csv
Processing sub-OE020 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Hellerup_fixations.csv
Processing sub-OE018 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE018/ses-Hellerup_fixations.csv
Processing sub-OE012 / ses-Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE012/ses-Norreport_fixations.csv
Processing sub-OE021 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE021/ses

## Temporal Alignment of Fixations with Video Frames

In [4]:
def align_fixations(session_path, fixations):

    datapicker = create_datapicker(path=session_path, schema=build_schema)
    dataset = load_dataset(datapicker.selected_path, schema=build_schema)

    gaze = dataset.streams.PupilLabs.PupilGaze.data

    gaze = (
        gaze.groupby(level=0)[["GazeX", "GazeY"]]
        .mean()
        .rename(columns={"GazeX": "x_mean", "GazeY": "y_mean"})
    )

    decoded_frames = dataset.streams.PupilLabs.DecodedFrames.data
    decoded_frames = decoded_frames[decoded_frames.Value > 0]

    gaze = gaze.reindex(decoded_frames.index, method="nearest")

    gaze["frame"] = np.arange(len(gaze))

    aligned = pd.merge_asof(
        fixations.sort_values("mid_time"),
        gaze,
        left_on="mid_time",
        right_index=True,
        direction="nearest"
    )

    return aligned

# Baseline 1: CLIP Model Initialization

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",use_safetensors=True).to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Urban Semantic Label Set (COCO + Extensions)

First I started only with the COCO labels, but then quickly realized that the crops were getting very wrong labels so I decided to add road-like labels as per the below </p>

extra_classes = ["floor", "ground", "pavement", "road", "sidewalk"] </p>

Then, labelling got slightly better (from what I could see in the 5 debug crops per participant), however I felt that the COCO labels were missing a lot of key urban gaze like labels, as so I added more like the below (in addition to what I already previously added):

extra_classes = ["road", "sidewalk", "pavement", "ground", "floor",</p>
    "building", "wall", "tree", "grass", "sky", "pole", "trash can", "shop"]</p>

like this the labeling reached a good enough point for me. Knowing that I didnt want to touch on the already defined COCO labels, since that would be a lot of my own bias.

In [6]:
# ===== LOAD COCO LABELS =====
ann_file = "/home/s243636/master-thesis/notebooks/instances_val2017.json"

coco = COCO(ann_file)
cats = coco.loadCats(coco.getCatIds())

# base classes
class_names = [c["name"] for c in cats]
print(class_names)

# extend class names
extra_classes = [
    "road", "sidewalk", "pavement", "ground", "floor",
    "building", "wall", "tree", "grass", "sky", "pole", "trash can", "shop"
]
class_names += extra_classes

# then build labels from class_names
labels = [f"a photo of a {name}" for name in class_names]

print(len(class_names))

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
93


## CLIP Inference on Fixation-Centered Image Crops

In [7]:
# ===== CLIP FUNCTION =====
def classify_crop(crop):
    image = Image.fromarray(cv.cvtColor(crop, cv.COLOR_BGR2RGB))

    inputs = processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

    return class_names[probs.argmax().item()]

## Frame-wise Fixation Classification Using CLIP

In [8]:
def process_video_with_clip(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        for _, gaze in gaze_rows.iterrows():
            x, y = int(gaze["x_mean"]), int(gaze["y_mean"])

            crop = frame[
                max(0, y-120):y+120,
                max(0, x-120):x+120
            ]

            label = classify_crop(crop)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

## Baseline Implementation 1: CLIP-Based Fixation Classification

In [9]:
data_root = "/mnt/raid/emotional_data_raquel/data"
fix_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip"

os.makedirs(output_root, exist_ok=True)

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)
    participant_label = f"sub-{participant}"

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant_label}/{session_name}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant_label}/{session_name}")

        # --- OUTPUT DIR ---
        save_dir = os.path.join(output_root, participant_label)
        os.makedirs(save_dir, exist_ok=True)

        # =========================
        # DEBUG CROPS (first 5)
        # =========================
        cap = cv.VideoCapture(video_path)

        for i, (_, row) in enumerate(aligned.head(5).iterrows()):
            try:
                frame_id = int(row["frame"])
                cx = int(row["x_mean"])
                cy = int(row["y_mean"])

                cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()

                if not ret:
                    continue

                patch_size = 120
                crop = frame[
                    max(0, cy - patch_size): cy + patch_size,
                    max(0, cx - patch_size): cx + patch_size
                ]

                if crop.size == 0:
                    continue

                label = classify_crop(crop)  # CLIP function

                plt.figure()
                plt.imshow(cv.cvtColor(crop, cv.COLOR_BGR2RGB))
                plt.title(label)
                plt.axis("off")

                plt.savefig(os.path.join(save_dir, f"debug_{session_name}_{i}.png"), bbox_inches="tight")
                plt.close()

            except Exception as e:
                print(f"Debug error: {e}")

        cap.release()

        # =========================
        # FULL CLIP PROCESSING
        # =========================
        try:
            results = process_video_with_clip(video_path, aligned)
        except Exception as e:
            print(f"CLIP error {participant_label}/{session}: {e}")
            continue

        # --- SAVE CSV ---
        save_path = os.path.join(save_dir, f"ses-{session_name}_clip.csv")

        pd.DataFrame(results).to_csv(save_path, index=False)

        print(f"Saved → {save_path}")

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE011/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE011/ses-Nordhavn_clip.csv
Processing sub-OE015/Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE015/ses-Norreport_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE019/Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE019/ses-Hellerup_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE010/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE010/ses-Nordhavn_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE024/Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE024/ses-Nordhavn_clip.csv
Attempting to automatically corr

# Baseline 2: CLIP (Gaussian) Model Initialization

In [40]:
def gaussian_focus(frame, cx, cy, sigma=80):

    h, w = frame.shape[:2]

    # create coordinate grid
    x = np.arange(w)
    y = np.arange(h)
    xx, yy = np.meshgrid(x, y)

    # gaussian mask
    mask = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    mask = mask[..., None]  # make 3-channel

    # blur full image
    blurred = cv.GaussianBlur(frame, (51, 51), 0)

    # combine sharp + blurred
    focused = frame * mask + blurred * (1 - mask)

    return focused.astype(np.uint8)

In [41]:
def process_video_with_clip_gaussian(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        for _, gaze in gaze_rows.iterrows():
            x, y = int(gaze["x_mean"]), int(gaze["y_mean"])

            focused = gaussian_focus(frame, x, y)

            label = classify_crop(focused)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

Switched from applying Gaussian to the full frame to using it within a cropped region around the fixation. The full-frame approach gave too much weight to the overall scene (e.g., pavement, instead of understanding that the user was actually focusing on the bikes within the pavement), so adding a crop helps focus on the actual gaze target while still keeping some surrounding context through the Gaussian.

In [42]:
def process_video_with_clip_gaussian_crop(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        for _, gaze in gaze_rows.iterrows():
            x, y = int(gaze["x_mean"]), int(gaze["y_mean"])

            patch_size = 200

            # --- STEP 1: crop (locality) ---
            crop = frame[
                max(0, y-patch_size):y+patch_size,
                max(0, x-patch_size):x+patch_size
            ]

            if crop.size == 0:
                continue

            # --- STEP 2: gaussian INSIDE crop ---
            focused = gaussian_focus(crop, patch_size, patch_size)

            # --- STEP 3: classify ---
            label = classify_crop(focused)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

In [43]:
# ===== LOAD COCO LABELS =====
ann_file = "/home/s243636/master-thesis/notebooks/instances_val2017.json"

coco = COCO(ann_file)
cats = coco.loadCats(coco.getCatIds())

# base classes
class_names = [c["name"] for c in cats]
print(class_names)

# extend class names
extra_classes = [
    "road", "sidewalk", "pavement", "ground", "floor",
    "building", "wall", "tree", "grass", "sky", "pole", "trash can", "shop"
]
class_names += extra_classes

# labels here more aligned with gaze like data
labels = [f"a person is looking at a {name}" for name in class_names]

print(len(class_names))

loading annotations into memory...
Done (t=0.31s)
creating index...
index created!
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
93


## Baseline Implementation 2: Gaussian CLIP-Based Fixation Classification

In [ ]:
data_root = "/mnt/raid/emotional_data_raquel/data"
fix_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/eyetracking/clip_gaussian"

os.makedirs(output_root, exist_ok=True)

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)
    participant_label = f"sub-{participant}"

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant_label, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant_label}/{session_name}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant_label}/{session_name}")

        # --- OUTPUT DIR ---
        save_dir = os.path.join(output_root, participant_label)
        os.makedirs(save_dir, exist_ok=True)

        # =========================
        # DEBUG CROPS (first 5)
        # =========================
        cap = cv.VideoCapture(video_path)
        patch_size = 200

        for i, (_, row) in enumerate(aligned.head(5).iterrows()):
            try:
                frame_id = int(row["frame"])
                cx = int(row["x_mean"])
                cy = int(row["y_mean"])

                cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
                ret, frame = cap.read()

                if not ret:
                    continue

                # ✅ define crop properly
                crop = frame[
                    max(0, cy - patch_size): cy + patch_size,
                    max(0, cx - patch_size): cx + patch_size
                ]

                if crop.size == 0:
                    continue

                # ✅ apply gaussian on crop
                focused = gaussian_focus(crop, patch_size, patch_size)
                label = classify_crop(focused)

                plt.figure()
                plt.imshow(cv.cvtColor(focused, cv.COLOR_BGR2RGB))
                plt.title(label)
                plt.axis("off")

                plt.savefig(os.path.join(save_dir, f"debug_{session_name}_{i}.png"), bbox_inches="tight")
                plt.close()

            except Exception as e:
                print(f"Debug error: {e}")

        cap.release()

        # =========================
        # FULL CLIP PROCESSING
        # =========================
        try:
            results = process_video_with_clip_gaussian_crop(video_path, aligned)
        except Exception as e:
            print(f"CLIP error {participant_label}/{session}: {e}")
            continue

        # --- SAVE CSV ---
        save_path = os.path.join(save_dir, f"ses-{session_name}_clip_gaussian.csv")

        pd.DataFrame(results).to_csv(save_path, index=False)

        print(f"Saved → {save_path}")

_____________

# Model 1: DeepLabV3 Model Initialization

In [6]:
# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load weights (this includes metadata = labels)
weights = DeepLabV3_ResNet101_Weights.DEFAULT

model = models.segmentation.deeplabv3_resnet101(weights=weights)
model = model.to(device)
model.eval()

# # Preprocessing
# preprocess = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((520, 520)),  # required size
#     transforms.ToTensor(),
# ])

# Get preprocessing FROM THE MODEL (important)
preprocess = weights.transforms()

# Get labels FROM THE MODEL (no manual list)
labels = weights.meta["categories"]

In [7]:
def deeplab_gaze_label(frame, x, y, r=100):
    h, w = frame.shape[:2]

    x1 = max(0, x - r)
    y1 = max(0, y - r)
    x2 = min(w, x + r)
    y2 = min(h, y + r)

    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return None, None

    input_tensor = preprocess(crop).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)["out"][0]

    seg = output.argmax(0).cpu().numpy()

    cy = seg.shape[0] // 2
    cx = seg.shape[1] // 2

    class_id = int(seg[cy, cx])
    label = labels[class_id]

    return class_id, label

In [ ]:
def process_video_with_deeplab(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        for _, gaze in gaze_rows.iterrows():

            x = int(gaze["x_mean"])
            y = int(gaze["y_mean"])

            # --- DeepLab instead of CLIP ---
            class_id, label = deeplab_gaze_label(frame, x, y)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "deeplab_id": class_id,
                "deeplab_label": label
            })

    cap.release()
    return results

In [ ]:
data_root = "/mnt/raid/emotional_data_raquel/data"
aligned_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/deeplab"

participants = os.listdir(data_root)

for participant in participants:

    participant_path = os.path.join(data_root, participant)
    if not os.path.isdir(participant_path):
        continue

    participant_label = f"sub-{participant}"

    sessions = os.listdir(participant_path)

    for session in sessions:

        print(f"Processing {participant_label} - {session}")

        session_path = os.path.join(participant_path, session)

        # --- video path ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        # --- aligned fixation file ---
        aligned_path = os.path.join(
            aligned_root,
            participant_label,
            session,
            "aligned_fixations.csv"   # <-- adjust name if needed
        )

        if not os.path.exists(video_path) or not os.path.exists(aligned_path):
            print("Missing data, skipping...")
            continue

        # --- load aligned ---
        aligned = pd.read_csv(aligned_path)

        # --- run DeepLab ---
        results = process_video_with_deeplab(video_path, aligned)

        df = pd.DataFrame(results)

        # --- save ---
        out_dir = os.path.join(output_root, participant_label, session)
        os.makedirs(out_dir, exist_ok=True)

        df.to_csv(os.path.join(out_dir, "deeplab.csv"), index=False)

        print(f"Saved: {participant_label} - {session}")